In [ ]:
# 1. Download data
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.preprocessing import StandardScaler

try:
    from hmmlearn.hmm import GaussianHMM
except ImportError:
    GaussianHMM = None

warnings.filterwarnings("ignore", category=FutureWarning)
plt.style.use("seaborn-v0_8-whitegrid")

TRADING_DAYS = 252
DATA_PERIOD = "20y"
train_window = 756
refit_step = 22

MARKETS = {
    "DJI": {"underly": "^DJI", "vol": "^VXD"},
    "NSDQ": {"underly": "^NDX", "vol": "^VXN"},
}


def download_close(ticker: str, period: str = DATA_PERIOD) -> pd.Series:
    raw = yf.download(ticker, period=period, auto_adjust=True, progress=False)
    if raw.empty or "Close" not in raw:
        raise ValueError(f"No close data returned for {ticker}")

    close = raw["Close"]
    if isinstance(close, pd.DataFrame):
        close = close.iloc[:, 0]
    return close.astype(float)


def download_market(underly_ticker: str, vol_ticker: str) -> pd.DataFrame:
    return pd.concat(
        {
            "UNDERLY": download_close(underly_ticker),
            "VOL": download_close(vol_ticker),
        },
        axis=1,
    ).sort_index()


prices_by_market = {
    name: download_market(config["underly"], config["vol"])
    for name, config in MARKETS.items()
}


In [ ]:
# 2. Feature engineering
# Use 22 trading days as the monthly window and 252 trading days per year.
def build_features(prices: pd.DataFrame) -> pd.DataFrame:
    df = prices.copy()
    df["ret"] = np.log(df["UNDERLY"] / df["UNDERLY"].shift(1))
    df["RV_22"] = df["ret"].rolling(22).std() * np.sqrt(TRADING_DAYS) * 100
    df["RV_63"] = df["ret"].rolling(63).std() * np.sqrt(TRADING_DAYS) * 100
    df["RV_63_22"] = df["RV_63"] - df["RV_22"]
    df["lagged_VRP"] = df["VOL"].shift(22) - df["RV_22"]
    df["trend_63"] = df["UNDERLY"].pct_change(63)
    df["drawdown_252"] = df["UNDERLY"] / df["UNDERLY"].rolling(252, min_periods=63).max() - 1
    df["sign_flip_22"] = np.sign(df["ret"]).ne(np.sign(df["ret"]).shift(1)).rolling(22).mean()
    return df


hmm_features = ["VOL", "RV_22", "RV_63", "RV_63_22", "lagged_VRP", "trend_63", "drawdown_252", "sign_flip_22"]
study_by_market = {
    name: build_features(prices).dropna(subset=hmm_features).copy()
    for name, prices in prices_by_market.items()
}

for name, study_df in study_by_market.items():
    print(f"{name} sample: {study_df.index.min().date()} to {study_df.index.max().date()} ({len(study_df):,} rows)")


In [ ]:
# 3. Training
# The HMM transition matrix is used internally by Viterbi decoding.
def label_high_low(data: pd.DataFrame, raw_labels: pd.Series) -> pd.Series:
    score = (
        data["RV_22"].groupby(raw_labels).mean().rank()
        + data["sign_flip_22"].groupby(raw_labels).mean().rank()
        - data["trend_63"].groupby(raw_labels).mean().rank()
    )
    high_label = score.idxmax()
    return pd.Series(np.where(raw_labels.eq(high_label), "high_vol", "low_vol"), index=raw_labels.index)


def hmm_regime(data: pd.DataFrame, features: list[str]) -> pd.Series:
    if GaussianHMM is None:
        raise ImportError("Install hmmlearn to run the HMM: pip install hmmlearn")

    hmm_data = data[features].replace([np.inf, -np.inf], np.nan).dropna()
    regimes = pd.Series(index=hmm_data.index, dtype="object", name="hmm")

    for train_end in range(train_window, len(hmm_data), refit_step):
        train_start = train_end - train_window
        test_end = min(train_end + refit_step, len(hmm_data))
        train = hmm_data.iloc[train_start:train_end]
        test = hmm_data.iloc[train_end:test_end]

        scaler = StandardScaler()
        x_train = scaler.fit_transform(train)
        x_test = scaler.transform(test)

        best_model = None
        best_score = -np.inf
        for seed in [7, 21, 42, 101]:
            model = GaussianHMM(
                n_components=2,
                covariance_type="diag",
                n_iter=500,
                min_covar=1e-3,
                random_state=seed,
            )
            try:
                model.fit(x_train)
                score = model.score(x_train)
            except Exception:
                continue

            if np.isfinite(score) and score > best_score:
                best_model = model
                best_score = score

        if best_model is None:
            continue

        train_states = pd.Series(best_model.predict(x_train), index=train.index)
        test_states = pd.Series(best_model.predict(x_test), index=test.index)
        train_labels = label_high_low(data.loc[train.index], train_states)
        high_state = train_states[train_labels.eq("high_vol")].mode().iloc[0]
        regimes.loc[test.index] = np.where(test_states.eq(high_state), "high_vol", "low_vol")

    return regimes.dropna()


regimes_by_market = {
    name: hmm_regime(study_df, hmm_features)
    for name, study_df in study_by_market.items()
}
study_by_market = {
    name: study_df.join(regimes_by_market[name])
    for name, study_df in study_by_market.items()
}


In [ ]:
# 4. Display
def plot_regime_shading(name: str, data: pd.DataFrame, regime: pd.Series) -> None:
    plot_df = data[["UNDERLY", "RV_22", "VOL"]].join(regime.rename("regime"), how="inner").dropna()
    fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True, gridspec_kw={"height_ratios": [2.2, 1]})

    axes[0].plot(plot_df.index, plot_df["UNDERLY"], color="black", linewidth=1.2, label=name)
    high_vol = plot_df["regime"].eq("high_vol")
    blocks = high_vol.ne(high_vol.shift()).cumsum()
    for _, block in plot_df[high_vol].groupby(blocks[high_vol]):
        axes[0].axvspan(block.index[0], block.index[-1], color="red", alpha=0.16, linewidth=0)
        axes[1].axvspan(block.index[0], block.index[-1], color="red", alpha=0.16, linewidth=0)

    axes[0].set_title(f"{name} with HMM high-vol regime shading")
    axes[0].set_ylabel(name)
    axes[0].legend(loc="upper left")

    axes[1].plot(plot_df.index, plot_df["RV_22"], color="tab:blue", linewidth=1.0, label="22d realized vol")
    axes[1].plot(plot_df.index, plot_df["VOL"], color="tab:orange", linewidth=1.0, alpha=0.75, label="implied vol")
    axes[1].set_ylabel("Annualized vol %")
    axes[1].legend(loc="upper left")
    plt.show()


for name in MARKETS:
    plot_regime_shading(name, study_by_market[name], regimes_by_market[name])
